# HRA-3D — Model Coverage on the Human Reference Atlas 3D Anatomy

_Investigation `hra-3d` — coder reproduction notebook._

**Question.** Which of the HRA's 1,730 Uberon-keyed anatomical structures (2,295 raw
ASCT+B-3D crosswalk rows, 114 source-organ GLBs) are covered by a
mechanistic model — at a single query's scale and at the full
1,096-model curated-corpus scale — and can we view that coverage/linkage
directly on the 3D anatomy?

Builds the 3D-anatomy side of the fetch-and-compare spine: the full
ASCT+B-3D crosswalk and a glomerulus FTU digital object, organ-
granularity model coverage crossed with the biomodel-DO index at both
query scale and full-corpus scale, model->AS->GLB-node spatial linkage,
and a materialized three.js viewer pack.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-human-atlas/viva-human-atlas').is_dir():
    REPO = Path('/home/runner/work/viva-human-atlas/viva-human-atlas')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from viva_human_atlas.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: HRA 3D Crosswalk — Full ASCT+B-3D Anatomical-Structure Tree (`hra-3d-crosswalk`)

**Question.** Does the HRA CDN's ASCT+B-3D models crosswalk load into a Step-driven
composite, giving the full 1,400+ anatomical-structure (AS) tree — not
just the ~81 per-organ reference-organ GLBs — with resolvable Uberon CURIEs
and source-organ GLB names?


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: hra-3d-crosswalk ===
STUDY = 'hra-3d-crosswalk'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: Model Coverage 3D — AS/Organ Coverage from BioModel-DO Annotations (`model-coverage-3d`)

**Question.** Of the HRA ASCT+B-3D crosswalk's 1,400+ anatomical structures (and ~81
reference organs), which are covered by a mechanistic model — and can
that coverage be viewed directly on the 3D anatomy?


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: model-coverage-3d ===
STUDY = 'model-coverage-3d'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: Spatial Linkage — Model -> AS -> GLB Scene-Node Links (`spatial-linkage`)

**Question.** For each biomodel-DO organ annotation, exactly which ASCT+B-3D crosswalk
GLB scene node(s) does it correspond to — precise enough for a 3D viewer
to color/label the right node per model?


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: spatial-linkage ===
STUDY = 'spatial-linkage'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: FTU Glomerulus — 3D Functional-Tissue-Unit Digital Object (`ftu-glomerulus`)

**Question.** Does the HRA CDN's glomerulus 3D functional-tissue-unit (FTU) digital
object load into a Step-driven composite, giving a sub-organ-scale 3D
asset (below the per-organ reference-organ GLBs) that could anchor
finer-grained model coverage?


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: ftu-glomerulus ===
STUDY = 'ftu-glomerulus'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: Corpus Coverage — Full 1,096-Model BioModels Catalog on HRA 3D Anatomy (`corpus-coverage`)

**Question.** Of the HRA ASCT+B-3D crosswalk's 1,730 Uberon-keyed anatomical structures
(across 114 source-organ GLBs), how many are covered once every curated
BioModels model — not just a narrow search query — is crossed against
them, and does full-corpus coverage reach organs a single-query search
misses (e.g. kidney)?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_human_atlas.composites.corpus_coverage_composite.corpus-coverage` | 0 | catalog_path=datasets/biomodel_corpus_catalog.json |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_human_atlas.composites.corpus_coverage_composite.corpus-coverage`** — `spec_viva_human_atlas_composites_corpus_coverage_composite_corpus_coverage` (a plain, editable dict)


_composite spec file for `viva_human_atlas.composites.corpus_coverage_composite.corpus-coverage` not found under `viva_human_atlas/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: corpus-coverage ===
STUDY = 'corpus-coverage'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: FTU Model Coverage — Do Existing BioModels Model HRA Functional Tissue Units? (`ftu-model-coverage`)

**Question.** Do existing BioModels model HRA functional tissue units (FTUs) -- Katy
Boerner's question -- and, for the FTUs that already have models, can HRA
CTpop (Cell Type Populations) cell-type counts parameterize them, per the
RUI -> CTpop -> model vision Peter Hunter and SPARC-heart describe?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_human_atlas.composites.ftu_coverage_composite.ftu-model-coverage` | 0 | catalog_path=datasets/biomodel_corpus_catalog.json |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_human_atlas.composites.ftu_coverage_composite.ftu-model-coverage`** — `spec_viva_human_atlas_composites_ftu_coverage_composite_ftu_model_coverage` (a plain, editable dict)


_composite spec file for `viva_human_atlas.composites.ftu_coverage_composite.ftu-model-coverage` not found under `viva_human_atlas/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: ftu-model-coverage ===
STUDY = 'ftu-model-coverage'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: CTpop -> Topp2000 Islet Parameterization — Beta-Cell Composition Shifts the Glucose Setpoint (`ctpop-islet-parameterization`)

**Question.** Does binding HRA CTpop islet cell-type composition to the Topp2000
beta-cell-mass model (BIOMD0000000341) change the predicted glucose
regulation -- realizing the RUI -> CTpop -> model parameterization vision
Katy Boerner / Peter Hunter / SPARC describe, for the pancreatic islet FTU
`ftu-model-coverage` already found covered?


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: ctpop-islet-parameterization ===
STUDY = 'ctpop-islet-parameterization'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: Blood Vasculature Network — VCCF transport graph to couple organ models (`blood-vasculature-network`)

**Question.** Can the HRA/VCCF vasculature data define a whole-body blood-transport
network — a directed heart -> organ -> heart circuit through named vessels —
that we can use to COUPLE independent organ simulations by advecting shared
blood-borne solutes (glucose, O2, insulin, ...) between them? And what does a
v1 blood-circulation simulation over that network look like?


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `viva_human_atlas.composites.vasculature_network_composite.blood-vasculature-network` | 0 | include_routes=True |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `viva_human_atlas.composites.vasculature_network_composite.blood-vasculature-network`** — `spec_viva_human_atlas_composites_vasculature_network_composite_blood_vasculature_network` (a plain, editable dict)


_composite spec file for `viva_human_atlas.composites.vasculature_network_composite.blood-vasculature-network` not found under `viva_human_atlas/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: blood-vasculature-network ===
STUDY = 'blood-vasculature-network'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Study: HRA Computational Model Atlas — organ selector + model-count gradient + subregion placement (`hra-atlas-browser`)

**Question.** Can we present the full HRA-3D model-coverage picture as an interactive
atlas — pick any of the 50 GLB-backed HRA organs, see it colored by how many
mechanistic models are associated with it, browse straight to those
BioModels — AND go one level deeper, placing models at the specific organ
SUBREGIONS (anatomical structures) their cell types / FTUs resolve to,
rather than distributing every model uniformly across the whole organ?


### Parameters


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: hra-atlas-browser ===
STUDY = 'hra-atlas-browser'
STUDY_DIR = REPO / 'studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

## Open decisions
- Which HRA organs/FTUs have ZERO mechanistic models (the modeling white-space)?
- Does organ-granularity coverage overstate AS-level coverage — can we get finer?
- Can we spatially link model RESULTS (steady-state values) to AS, not just presence?
